# Does a glioma feed its own growth? A neuron–glioma feedback loop and what a growth curve cannot tell

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/06_neuron_glioma_feedback.ipynb)

*In Google Colab, the first code cell clones the public repository and puts its package on the path; Colab's own NumPy, Matplotlib and ipywidgets are used, not the locked versions.*
**Question:** If neural activity can speed up glioma growth, and glioma cells can in turn increase that
activity, how does the feedback change growth compared with activity that is simply imposed? And can
a growth curve tell the two apart?
**Evidence:** A synthetic, coarse-grained construction in days on a constructed millimetre plane
(Table 1, Figures 1–3 and two interactive explorers). No coefficient is estimated from MRI, EEG, MEG or
molecular data.

**Why this matters.** Cancer neuroscience has shown that gliomas are wired into the brain. Neurons form
synapses onto glioma cells, and neural activity promotes glioma growth
([Venkatesh et al. 2019](https://doi.org/10.1038/s41586-019-1563-y); [Venkataramani et al. 2019](https://doi.org/10.1038/s41586-019-1564-x)). In the other direction,
gliomas make the surrounding cortex more excitable, which is one reason why seizures are common in
patients with glioma ([Buckingham et al. 2011](https://doi.org/10.1038/nm.2453); [McAlpine et al. 2025](https://doi.org/10.1038/s41593-025-02149-0)). Put
together, this is a *loop*: more tumour, more activity, more growth. Venkatesh et al. call it a positive
feedback mechanism. A loop matters clinically because it could make a tumour accelerate itself, and
because it suggests where to intervene: on the activity, on the tumour, or on the connection between
them. Human recordings link the degree of tumour–brain functional connectivity to survival and to
language performance ([Krishna et al. 2023](https://doi.org/10.1038/s41586-023-06036-1)). Work from 2025 and 2026 shows that the loop
is richer than one excitatory arrow: gliomas receive input from many transmitter systems, inhibitory
circuits can restrain adult gliomas, and the same transmitter can promote growth in one glioma subtype
but not in another (section 1). But loops are hard to reason about
informally, because cause and effect run both ways. This notebook builds the smallest model of such a
loop and asks what it does, and what a growth curve alone could reveal about it.

**Aims.** After working through this notebook you should be able to
1. summarize, with references, the experimental evidence for each direction of the neuron–glioma loop,
   and the settings (tumour type, species, model system) in which it was obtained;
2. write down a two-field model of the loop and say what each term, symbol and unit means;
3. compare three nested assumptions (no coupling, one-way coupling and feedback) and read their
   differences in a table, a time course and a phase plane;
4. explain why a one-way activity effect is exactly indistinguishable from a faster-growing tumour
   when only density is observed;
5. say which additional measurements would be needed to tell the explanations apart, and why no
   clinical conclusion follows from the model.

**Two tracks.** Most sections have a **clinical and biological track**, which connects the model to
glioma biology, electrophysiology and clinical observation, and a **modeling track**, which gives the
equations, derivations and checks. Read either or both; the code cells and figures are shared.
Acronyms and terms from both fields are explained in the glossary after the next section, and all
references are listed, with links, at the end.

**Every result below is synthetic.** This is a thought experiment, not a reproduction of the cited
papers, a neural-circuit simulator or a patient forecast.

**Place in the perspective:** a slow, day-scale growth–signal construction and the main talk example.
Notebooks 07 and 08 separately teach potassium clearance and extracellular field measurements on a
scale of seconds. No coupling between those examples and this signal has been established.

## AI contribution and verification

**Question.** Neurons can form synapses onto glioma cells and neural activity can promote glioma growth,
while gliomas can make surrounding circuits more excitable. This notebook builds the smallest model of
that loop and asks:
- *Effect of the loop:* for the same imposed activity, how much faster does density grow when the
  tumour can also raise the activity signal?
- *Where:* in space, where does the feedback act, and how does that depend on where activity is
  imposed?
- *Inference:* can a density curve distinguish an activity effect from a tumour that simply grows
  faster?

**Clinical and biological track.** Read the three questions as: if a tumour makes its surroundings more
excitable, does that make it grow faster than activity alone would; does it matter whether the tumour
sits near highly active cortex; and could serial scans that measure only tumour size ever show that
activity is involved?

**Modeling track.** The model couples a reaction–diffusion equation for density to a relaxation
equation for a signal. The three questions become: the effect of the feedback coefficient β at fixed
coupling κ; the spatial pattern of the signal increase $a-b$; and whether the parameters are
identifiable from density alone, which they are not (an exact equivalence, shown in section 3).

**AI contribution.**
- *Trained imaging AI:* none in this notebook.
- *Generative research assistance* (development only; records in `docs/ai-use/`): Codex desktop
  (6 October 2026; model snapshot not captured) helped implement the phenomenological model, nested
  controls, sensitivity analysis and explanations after the project requested cancer neuroscience
  ([record](../docs/ai-use/2026-10-06-biology-modeling.md)). On 10 October 2026 a Cursor agent (model
  and settings not captured) wrote an expanded version: the well-mixed solver
  `well_mixed_neuron_glioma` in `glioma_ai.biology`, both explorers in `glioma_ai.biology_interactive`,
  their tests and the explanatory text. On 13 October 2026 a Cursor agent (model and settings not
  captured) restructured the text into two tracks, added the glossary and the additional references,
  on request, and then, on a further request, recent (2025–2026) papers and relevant papers co-authored
  by Bergen glioma researchers; every reference was checked against Crossref and its PubMed abstract.
- *Conventional code* (rerun every time, no model): the explicit spatial solver, the fourth-order
  well-mixed solver and the checks. Rebuilding the notebook calls no language model.

**Design choices awaiting scientific review.** The project asked for cancer neuroscience as the main
talk example, presented didactically with interactive demonstrations. Within that scope, the following
are assistant choices, with their reasons:
- *An abstract signal index a in [0, 1] instead of spikes or voltages:* growth acts over days, so a
  slow lumped index keeps the time scales consistent. It is not a firing rate.
- *Growth multiplied by (1 + κa):* the simplest way for the signal to modulate growth; κ = 0 recovers
  the uncoupled model exactly.
- *Feedback β c (1 − a):* density raises the signal, which cannot exceed 1.
- *Values D = 0.2 mm²/day, ρ = 0.04/day, κ = 1.5, β = 0.12/day, τ = 4 days:* chosen so that the three
  cases separate within 20 days on a 1 mm grid. Not estimated.
- *A well-mixed (no space) version:* added for teaching because it has exact limits and can be drawn
  in a phase plane.
- *The selection of references and the glossary entries.*

**Verification.**
- *Nested limits:* with κ = β = 0 the spatial solver reproduces the plain reaction–diffusion solver
  bit for bit; in the well-mixed model it equals the exact logistic curve (Table 1).
- *Confounding identity:* with β = 0 and a constant signal, one-way coupling is exactly a logistic
  curve with $\rho_{\text{eff}}=\rho(1+\kappa b)$, in space and without it.
- *Numerics:* halving the spatial time step changes the final density integral by about 0.1%; the
  well-mixed solver converges at fourth order (tested).
- *Phase plane:* the feedback trajectory stays within 0.02 of the analytic signal nullcline.
- *Not verified here:* any coefficient, the shape of the coupling, agreement with tissue or
  electrophysiological data, and any MRI appearance.

**Evidence status.**
- *Table 1, Figures 1–3 and the explorers:* synthetic constructions in stated units.
- *Development records:* actual contributions, summarized retrospectively and pending human
  scientific review. Comparative benefit of the assistance is unmeasured.

These roles describe development and provenance. [Assistance records](../docs/ai-workflow.md) retain
the review status; record completeness is separate from scientific quality.

## Glossary: acronyms and key terms

Refer back to these tables as needed. Terms are grouped by field; within each group they are roughly
in the order in which they appear.

**Clinical, biological and electrophysiological terms**

| Term | Meaning |
| --- | --- |
| Cancer neuroscience | The study of how the nervous system regulates cancer and how cancers and their treatments change the nervous system ([Monje et al. 2020](https://doi.org/10.1016/j.cell.2020.03.034); [Winkler et al. 2023](https://doi.org/10.1016/j.cell.2023.02.002)). |
| Glioma, HGG, LGG | Glioma: a primary brain tumour of glial character. HGG and LGG: high-grade and low-grade glioma, a distinction based on histological and molecular features that broadly predicts aggressiveness. |
| GBM, IDH | Glioblastoma, which in the 2021 WHO classification ([Louis et al. 2021](https://doi.org/10.1093/neuonc/noab106)) denotes IDH-wildtype diffuse astrocytic tumours of grade 4 in adults. IDH: isocitrate dehydrogenase, whose mutation defines a separate group of gliomas. |
| DIPG | Diffuse intrinsic pontine glioma, a paediatric high-grade glioma of the brainstem, used in several of the cited experiments. |
| DMG, hemispheric HGG | Diffuse midline glioma (including H3K27M-altered DIPG) arises in midline structures such as the pons and thalamus; hemispheric high-grade gliomas, such as IDH-wildtype glioblastoma, arise in the cerebral hemispheres. Several 2025–2026 studies found that these subtypes respond differently to neural input. |
| Pilocytic astrocytoma | The most common brain tumour in children, usually low grade. |
| Synapse | A contact through which a neuron passes a signal to another cell, usually by releasing a neurotransmitter. |
| Glutamate, AMPA receptor | Glutamate is the main excitatory neurotransmitter of the brain. AMPA receptors are glutamate-gated ion channels; neuron–glioma synapses act through them. |
| GABA, GABA$_A$ receptor | The main inhibitory neurotransmitter of the adult brain and its ion-channel receptor. Whether GABA depolarizes or hyperpolarizes a cell depends on its internal chloride concentration, which the NKCC1 transporter raises. |
| Acetylcholine, muscarinic receptor (CHRM) | A neuromodulatory transmitter released by long-range projections, for example from the basal forebrain and brainstem; CHRM1 and CHRM3 are G-protein-coupled receptors for it. |
| Dopamine receptor D1 (DRD1) | A receptor for the neuromodulator dopamine, also expressed by glioblastoma cells ([Xue et al. 2024](https://doi.org/10.1186/s13046-024-02947-7)). |
| Neuron–glioma synapse | A synapse in which the neuron is presynaptic and a glioma cell is postsynaptic ([Venkatesh et al. 2019](https://doi.org/10.1038/s41586-019-1563-y); [Venkataramani et al. 2019](https://doi.org/10.1038/s41586-019-1564-x)). |
| Depolarization | A rise in the electrical potential across a cell membrane. Depolarizing glioma cells promoted their proliferation in mice (Venkatesh et al. 2019). |
| Gap junction | A channel that directly connects the interiors of neighbouring cells, so that ions and small molecules can pass. |
| Tumour microtubes | Long membrane protrusions by which astrocytic glioma cells invade and interconnect into a network ([Osswald et al. 2015](https://doi.org/10.1038/nature16071)). Not the same as the microtubules of the cytoskeleton. |
| NLGN3, ADAM10 | Neuroligin-3, a synaptic protein that is released in an activity-dependent way and promotes glioma proliferation ([Venkatesh et al. 2015](https://doi.org/10.1016/j.cell.2015.04.012)); ADAM10, the enzyme that cleaves and releases it ([Venkatesh et al. 2017](https://doi.org/10.1038/nature24014)). |
| PI3K–mTOR | An intracellular growth-signalling pathway activated by NLGN3 in glioma cells; mTOR signalling in neurons is also implicated in glioma-induced hyperexcitability ([Goldberg et al. 2025](https://doi.org/10.1016/j.neuron.2024.12.026)). |
| PDGFRα, Src, ERK | A receptor tyrosine kinase and downstream kinases through which glutamate receptors can drive tumour-cell proliferation without depolarization ([Anastasaki et al. 2025](https://doi.org/10.1016/j.neuron.2025.08.005)). |
| TGF-β | Transforming growth factor beta, a signalling protein that promotes tumour microtube formation in glioblastoma through thrombospondin-1 ([Joseph et al. 2022](https://doi.org/10.1093/neuonc/noab212)). |
| GAP43 | Growth-associated protein 43, involved in axon regeneration; it supports tumour microtubes (Osswald et al. 2015) and connections through which astrocytes transfer mitochondria to glioblastoma cells ([Watson et al. 2023](https://doi.org/10.1038/s43018-023-00556-5)). |
| C1QL1, synaptic pruning | A protein secreted by glioblastoma cells that expands tumour microtubes and prunes normal synapses, favouring neuron–glioma synapses ([Ding et al. 2026](https://doi.org/10.1158/2159-8290.CD-25-1123)). Pruning is the removal of existing synapses. |
| Kv4.2 (*KCND2*) | A voltage-gated potassium channel expressed by OPC-like glioma cells at the tumour border, which raises extracellular potassium and neuronal excitability ([Zhang et al. 2025](https://doi.org/10.1016/j.neuron.2024.10.016)); compare notebook 07. |
| Excitability, hyperexcitability | How readily neurons fire in response to input; hyperexcitability means abnormally readily, as in the cortex surrounding many gliomas. |
| Epileptiform activity, seizure | Abnormal, synchronous neural discharges; a seizure is the clinical event they can cause. Seizures are a common symptom of glioma. |
| xCT (system x$_c^-$) | A cystine–glutamate exchanger through which glioma cells release glutamate, contributing to peritumoral hyperexcitability ([Buckingham et al. 2011](https://doi.org/10.1038/nm.2453); [Hatcher et al. 2020](https://doi.org/10.1172/jci133316)). |
| Interneuron | An inhibitory neuron; losing interneurons removes a brake on cortical activity. |
| Antiseizure (antiepileptic) medication | Drugs that prevent seizures, such as levetiracetam (which acts mainly on the synaptic vesicle protein SV2A) and benzodiazepines such as lorazepam (which enhance GABA$_A$ receptor signalling). |
| Optogenetics | Switching selected neurons on or off with light, after genetically introducing light-sensitive channels. |
| Retrograde (rabies) tracing | Using a modified rabies virus that crosses synapses backwards to label the neurons that connect to a starting cell population, here tumour cells. |
| Xenograft, PDOX | Human tumour cells implanted into an animal, usually an immunodeficient mouse; patient-derived orthotopic xenografts (PDOX) are implanted into the corresponding brain region. |
| Organoid | A three-dimensional tissue culture that develops some of the structure of an organ; brain organoids can be co-cultured with tumour cells to study invasion ([Zhou et al. 2026](https://doi.org/10.1093/neuonc/noaf271)). |
| Patch clamp | Recording the electrical activity of a single cell with a glass micropipette, here in human brain tissue removed during surgery. |
| EEG, MEG, ECoG | Electro- and magnetoencephalography measure brain activity from the scalp; electrocorticography records directly from the brain surface, for example during surgery. |
| Functional connectivity | Statistical dependence between activity recorded at different sites. It is not a count of synapses. |
| TSP-1 | Thrombospondin-1, a synapse-promoting factor secreted by glioblastoma cells in functionally connected tumour regions ([Krishna et al. 2023](https://doi.org/10.1038/s41586-023-06036-1)). |

**Modeling terms**

| Term | Meaning |
| --- | --- |
| Reaction–diffusion model | A partial differential equation (PDE) in which a quantity spreads (diffusion) and is produced or removed locally (reaction); a standard model of glioma growth and invasion ([Swanson et al. 2003](https://doi.org/10.1016/j.jns.2003.06.001)). Notebook 02 introduces it. |
| Logistic growth, carrying capacity | Growth at rate ρ that slows as density approaches a maximum; here the maximum (carrying capacity) is normalized to 1. |
| $c$ (normalized density) | Tumour density relative to the carrying capacity, between 0 and 1. Not MRI intensity or a cell count. |
| $a$ (signal index) | A slow, dimensionless, activity-associated growth-signal index between 0 and 1. Not a firing rate, voltage or MRI signal. |
| $b$ (external drive) | The signal level that would hold without the tumour; prescribed, and allowed to vary in space. |
| $D$, ρ | Diffusion coefficient (mm²/day) and proliferation rate (1/day) of the tumour cells. |
| κ (coupling) | How strongly the signal multiplies growth; dimensionless. κ = 0 removes the effect of the signal on growth. |
| β (feedback) | How fast tumour density raises the signal (1/day). β = 0 removes the tumour's effect on the signal. |
| τ (relaxation time) | The time constant (days) with which the signal returns to the drive in the absence of tumour. |
| Positive feedback loop | A circuit in which the output reinforces its own cause: here density raises the signal, which raises growth. |
| Nested models | Models that are special cases of each other, obtained by setting parameters to zero: baseline ⊂ one-way ⊂ feedback. |
| ODE, PDE | Ordinary differential equation (change in time only) and partial differential equation (change in time and space). |
| Well-mixed | One location with no spatial structure; diffusion drops out. |
| Effective rate $\rho_{\text{eff}}$ | The single growth rate that reproduces a model with constant signal: $\rho(1+\kappa b)$. |
| Nullcline | The curve in the phase plane on which one variable does not change; here the signal nullcline, where $da/dt=0$. |
| Phase plane | A plot of one state variable against the other, here $a$ against $c$, in which a simulation becomes a trajectory. |
| Fourth-order Runge–Kutta (RK4) | A standard ODE method whose error falls 16-fold when the time step is halved. |
| Explicit Euler step | The simplest time step: the next value is the current value plus the time step times the current rate. Used for the spatial model. |
| Step halving | Rerunning with half the time step to check that the result does not depend on it. |
| No-flux boundary | A boundary through which no density can leave the plane. |
| Density integral | The sum of normalized density times grid-cell area, in mm² on this 2D plane; neither a cell number nor a clinical volume. |
| Sensitivity analysis | Rerunning with different prescribed parameters to see how results change. It is not parameter estimation. |
| Identifiability, confounding | Whether parameters can be determined from a given kind of data ([Raue et al. 2009](https://doi.org/10.1093/bioinformatics/btp358)). Two explanations are confounded when they produce the same data. |
| Synthetic construction | Inputs and parameters prescribed by the modeler, not measured. |

## 1. Biological background

**Clinical and biological track.**
- *Activity drives growth.* In a xenograft model of paediatric glioblastoma, optogenetically stimulated
  cortical neurons promoted high-grade glioma proliferation and growth; the activity-regulated synaptic
  protein NLGN3 was identified as the leading mitogen, and NLGN3 expression in human high-grade glioma
  correlated negatively with overall survival ([Venkatesh et al. 2015](https://doi.org/10.1016/j.cell.2015.04.012)). Xenografts of
  paediatric GBM, DIPG and adult GBM failed to grow in mice lacking *Nlgn3*, and blocking its release
  with ADAM10 inhibitors robustly blocked xenograft growth ([Venkatesh et al. 2017](https://doi.org/10.1038/nature24014)).
- *Gliomas receive synapses.* Two studies published together showed bona fide AMPA receptor-dependent
  synapses from neurons onto glioma cells ([Venkatesh et al. 2019](https://doi.org/10.1038/s41586-019-1563-y);
  [Venkataramani et al. 2019](https://doi.org/10.1038/s41586-019-1564-x)). Depolarizing glioma cells promoted proliferation, and
  blocking electrochemical signalling inhibited xenograft growth and extended mouse survival
  (Venkatesh et al.). The synapses sit on tumour microtubes, the protrusions that connect glioma cells
  into a network ([Osswald et al. 2015](https://doi.org/10.1038/nature16071)). Genetic perturbation of AMPA receptors in
  glioma cells reduced invasiveness and growth, and the AMPA receptor antagonist perampanel reduced
  growth (Venkataramani et al.).
- *Gliomas raise activity.* Glioma cells release glutamate through the xCT exchanger, causing
  epileptiform hyperexcitability that spreads into adjacent tissue in mice
  ([Buckingham et al. 2011](https://doi.org/10.1038/nm.2453)). In a native mouse model, particular *PIK3CA* variants
  selectively initiated brain hyperexcitability and remodelled synapses, with the secreted protein glypican-3 driving both gliomagenesis and
  hyperexcitability ([Yu et al. 2020](https://doi.org/10.1038/s41586-020-1952-2)). In an immunocompetent mouse model, hyperexcitability
  progressed to seizures, preceded by loss of inhibitory interneurons, gain of xCT and microglial
  inflammation that later extended far beyond the tumour boundaries
  ([Hatcher et al. 2020](https://doi.org/10.1172/jci133316)).
- *Evidence in humans.* Intraoperative electrocorticography showed increased excitability in
  glioma-infiltrated cortex (Venkatesh et al. 2019). Patch-clamp recordings in tissue from adults
  found pyramidal neurons in high-grade glioma more excitable than in low-grade glioma, and increased
  neuron–glioma network activity in high-grade tissue increased glioma proliferation
  ([McAlpine et al. 2025](https://doi.org/10.1038/s41593-025-02149-0)). In recordings during language tasks, gliomas remodelled
  functional circuitry so that task-relevant responses engaged tumour-infiltrated cortex; regions with high
  functional connectivity to the rest of the brain are enriched for a synaptogenic glioblastoma
  subpopulation secreting thrombospondin-1, and higher tumour–brain functional connectivity was
  associated with shorter survival and poorer language performance
  ([Krishna et al. 2023](https://doi.org/10.1038/s41586-023-06036-1)).
- *Beyond glutamate: many inputs, near and far (2024–2025).* Rabies tracing showed that glioblastoma
  rapidly integrates into neural circuits across the mouse brain, with cholinergic neurons driving
  invasion; radiotherapy enhanced neuron–tumour connectivity by increasing neuronal activity, and
  combining activity inhibition with radiotherapy increased the therapeutic effect
  ([Tetzlaff et al. 2025](https://doi.org/10.1016/j.cell.2024.11.002)). Human glioblastoma organoids transplanted into mice received
  local and long-range inputs from several transmitter systems; acetylcholine induced long-lasting
  calcium oscillations and reprogramming toward a more motile state through the CHRM3 receptor, and the
  authors propose that transient activation of upstream neurons may lead to a long-lasting increase in
  tumour fitness ([Sun et al. 2025](https://doi.org/10.1038/s41586-025-08634-7)). In models of pilocytic astrocytoma, glutamate increased
  proliferation *without* changing membrane depolarization, through glutamate receptor–Src–PDGFRα
  signalling ([Anastasaki et al. 2025](https://doi.org/10.1016/j.neuron.2025.08.005)). Glioblastoma cells also depend on
  neuromodulator receptors: blocking or knocking down the dopamine receptor DRD1 inhibited invasion and
  xenograft growth ([Xue et al. 2024](https://doi.org/10.1186/s13046-024-02947-7)).
- *Inhibition and subtype (2025–2026).* In diffuse midline gliomas, GABAergic synapses *depolarize*
  tumour cells (because NKCC1 keeps their chloride high) and promote proliferation; the benzodiazepine
  lorazepam increased growth of DMG xenografts but not of hemispheric glioblastoma xenografts
  ([Barron et al. 2025](https://doi.org/10.1038/s41586-024-08579-3)). Retrospective clinical data showed longer overall survival in
  children with DMG who were taking levetiracetam, not seen in hemispheric high-grade glioma, and
  levetiracetam reduced GABAergic synaptic currents in DMG cells and DMG xenograft growth; the authors
  call for prospective clinical studies ([Barron et al. 2026](https://doi.org/10.1038/s41591-026-04646-6)). In adult gliomas, by
  contrast, input from local inhibitory interneurons acted as a tumour-suppressive signal: enhancing
  inhibitory tone restrained proliferation and prolonged survival in mice, while disrupting it
  accelerated growth, so that progression is governed "by a balance between growth-promoting and
  growth-restraining circuit influences" ([Uesaka et al. 2026](https://doi.org/10.1016/j.neuron.2026.08.008)).
- *More ways in which gliomas raise activity (2025–2026).* Tumour-associated neurons at the infiltrative
  margin lose dendritic spines and become hyperexcitable through mTOR-driven changes, which a single dose
  of an mTOR inhibitor reversed in their experimental model ([Goldberg et al. 2025](https://doi.org/10.1016/j.neuron.2024.12.026)). In epileptic glioblastoma,
  OPC-like tumour cells accumulate at the tumour–neuron interface and express the potassium channel
  Kv4.2, raising extracellular potassium and neuronal excitability ([Zhang et al. 2025](https://doi.org/10.1016/j.neuron.2024.10.016);
  potassium clearance is the topic of notebook 07). Glioblastoma-secreted C1QL1 expands tumour
  microtubes and prunes normal synapses, favouring malignant synapses and recurrence
  ([Ding et al. 2026](https://doi.org/10.1158/2159-8290.CD-25-1123)).
- *The tumour network and its partners (Bergen and collaborators).* TGF-β promotes tumour microtube
  formation and calcium communication in glioblastoma through thrombospondin-1; microtubes are abundant
  in glioblastoma but uncommon in IDH-mutant, 1p/19q-codeleted oligodendrogliomas, and inhibiting the
  pathway reduced microtube formation and invasion ([Joseph et al. 2022](https://doi.org/10.1093/neuonc/noab212)). Thrombospondin-1
  is the same synaptogenic factor that Krishna et al. found in functionally connected tumour regions.
  GAP43-dependent connections between glioblastoma cells and astrocytes transfer mitochondria, which
  increases respiration, proliferation and tumorigenicity ([Watson et al. 2023](https://doi.org/10.1038/s43018-023-00556-5); review:
  [Storevik et al. 2026](https://doi.org/10.1093/neuonc/noag038)). The tumour's network partners are therefore not only neurons.
- *Clinical context.* In 16,007 glioma patients in Denmark, Norway, Sweden and Wales, new psychotropic
  prescriptions rose from about seven months before diagnosis, peaked around diagnosis and stayed above
  those of matched comparators for two years, driven largely by antiepileptics, anxiolytics, hypnotics
  and sedatives ([Baxter et al. 2025](https://doi.org/10.1007/s11060-025-04996-0)). Many patients therefore take drugs that act on neural
  activity; whether any of them changes tumour growth is, outside the specific DMG evidence above, an
  open question.
- *Boundaries.* Much of this evidence comes from mice, xenografts, organoids and specific tumour types,
  including paediatric tumours, DMG and DIPG. McAlpine et al. found differences between low- and
  high-grade gliomas, and the GABA studies found opposite roles in different subtypes.
  Results from one setting should not be silently generalized to every glioma (classification:
  [Louis et al. 2021](https://doi.org/10.1093/neuonc/noab106); field reviews: [Monje et al. 2020](https://doi.org/10.1016/j.cell.2020.03.034);
  [Winkler et al. 2023](https://doi.org/10.1016/j.cell.2023.02.002)).
- *What the model abstracts.* The two arrows of the loop, activity → growth and tumour → activity,
  become two coefficients, κ and β. Everything between them (synapses, NLGN3, glutamate,
  acetylcholine, GABA, potassium, interneuron loss, inflammation) is lumped into one slow signal index
  that can only *promote* growth. The model cannot represent the growth-restraining inhibitory input
  described by Uesaka et al., or a transmitter whose effect changes sign between subtypes.

**Modeling track.** The tumour part is the reaction–diffusion model of glioma growth and invasion
([Swanson et al. 2003](https://doi.org/10.1016/j.jns.2003.06.001)) used in notebook 02. To it we add one slow variable that
multiplies growth and is raised by density. The form is chosen to be minimal and to contain the
uncoupled model as an exact special case, so that every addition can be tested against a known answer. Omitted on
purpose: explicit neurons and synapses, electrophysiology, immune dynamics, treatment and any MRI
forward model. The recent literature points to three extensions a modeler should keep in mind:
- *Signed coupling.* Inhibitory input that restrains growth (Uesaka et al. 2026) would need either a
  second, inhibitory signal or a coupling that can be negative; GABA's opposite roles in DMG and adult
  glioma mean that the sign itself can depend on subtype.
- *Memory.* "Transient activation ... may lead to a long-lasting increase in tumour fitness" (Sun et al.
  2025) is not captured by a signal that relaxes with τ = 4 days; it suggests a slow change of tumour
  cell state, as in notebook 05.
- *Signal is not voltage.* Glutamate can drive proliferation without depolarization
  (Anastasaki et al. 2025), which supports treating $a$ as a growth-signal index rather than as
  membrane potential or firing rate.
Treatment can also change the loop: radiotherapy increased neuronal activity and connectivity in
Tetzlaff et al. (2025), which in model terms would change $b$ or β during treatment.

## 2. The model: two fields and explicit assumptions

$$\frac{\partial c}{\partial t}=\nabla\cdot(D\nabla c)+\rho\,(1+\kappa a)\,c\,(1-c),\qquad
\frac{\partial a}{\partial t}=\frac{b-a}{\tau}+\beta\,c\,(1-a).$$

Read term by term:
- $\nabla\cdot(D\nabla c)$: tumour cells spread by diffusion, as in notebook 02.
- $\rho\,c(1-c)$: logistic growth towards a normalized carrying capacity of 1.
- $(1+\kappa a)$: the signal multiplies growth; κ = 0 switches this off.
- $(b-a)/\tau$: without tumour, the signal relaxes to an external drive $b$ with time constant τ.
- $\beta c(1-a)$: tumour density raises the signal; β = 0 switches this off.

c is normalized density. a is a **slow activity-associated growth-signal index**; it is not firing rate,
voltage or an MRI intensity. b is a prescribed external drive. The model does not resolve synapses or
specific growth factors.

| Quantity | Prescribed value | Unit |
| --- | --- | --- |
| D | 0.2 | mm²/day |
| ρ | 0.04 | 1/day |
| κ | 1.5 | dimensionless |
| β | 0.12 | 1/day |
| τ | 4 | days |
| Grid spacing | 1 | mm |
| Time step / horizon | 0.1 / 20 | days |

Density has no-flux boundaries; the signal has no transport. Both fields use their previous values in
each explicit step. A combined diffusion/reaction and signal bound preserves states between 0 and 1
without clipping.

**Three nested assumptions** are compared throughout, with everything else fixed:
1. **Baseline:** κ = β = 0, the plain reaction–diffusion model.
2. **One-way:** κ = 1.5, β = 0. The imposed signal modulates growth, but the tumour cannot change it.
3. **Feedback:** κ = 1.5, β = 0.12/day. Density also raises the signal.

**Clinical and biological track.** Think of $b$ as the activity a region would have anyway, higher in
busy cortex and lower elsewhere, and of $a$ as the growth-promoting activity the tumour cells actually
experience. *One-way* means that activity promotes growth but the tumour does not change it: the
situation of the optogenetic experiments, where activity is imposed from outside. *Feedback* adds the
second arrow: where the tumour is dense, it raises activity, for example by releasing glutamate or
removing inhibition. A value of κ = 1.5 means that a fully active region ($a=1$) would grow 2.5 times
as fast as a silent one; this is a teaching choice, not a measurement.

**Modeling track.** The system is a semilinear reaction–diffusion equation for $c$ coupled to an ODE
for $a$ at every grid point, since $a$ does not diffuse. The coupling is bilinear in $a$ and $c$ in
both directions. Setting κ = 0 decouples density from the signal; setting β = 0 makes the signal
independent of density, so that with $a(0)=b$ it stays at $b$. The units fix the meaning of the
parameters: ρ and β are rates, τ is a time, κ, $a$, $b$ and $c$ are dimensionless. The explicit
scheme has a step bound that keeps $c$ and $a$ in $[0,1]$; the solver refuses steps above it.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def repository_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)


if IN_COLAB and repository_root() is None:
    checkout = Path("/content/AI-assisted-glioma-research")
    if not checkout.is_dir():
        url = "https://github.com/arvidl/AI-assisted-glioma-research"
        subprocess.run(["git", "clone", "--depth", "1", url, str(checkout)], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(Path.cwd() / "src"))
if repository_root() is None:
    raise RuntimeError("Open this notebook in a checkout of the repository or in Google Colab.")

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from glioma_ai.biology import NeuronGliomaParameters, simulate_neuron_glioma
from glioma_ai.biology_examples import neuron_glioma_construction, neuron_glioma_example
from glioma_ai.biology_interactive import (
    draw_feedback_state,
    feedback_scenario,
    logistic_density,
    signal_nullcline,
)
from glioma_ai.growth import integrate_fisher_2d
from glioma_ai.notebook_display import show_figure

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
output = ROOT / "outputs/notebooks/neuron_glioma"
summary = neuron_glioma_example(output)
summary["final_density_integral_mm2"]

## 3. Remove space first: the well-mixed model

If c and a are the same everywhere, diffusion vanishes and two ordinary differential equations remain:

$$\frac{dc}{dt}=\rho(1+\kappa a)\,c(1-c),\qquad \frac{da}{dt}=\frac{b-a}{\tau}+\beta c(1-a).$$

Three facts follow without a computer:
- **Baseline (κ = 0).** The density equation is the logistic equation, with exact solution
  $c(t)=c_0e^{\rho t}/(1-c_0+c_0e^{\rho t})$.
- **One-way (β = 0) with $a(0)=b$.** Then $da/dt=0$, the signal stays at $b$, and the density is again
  logistic, but with $\rho_{\text{eff}}=\rho(1+\kappa b)$. **A one-way activity effect is
  indistinguishable from a faster-growing tumour.**
- **Feedback (β > 0).** Setting $da/dt=0$ gives the *signal nullcline*
  $a=(b/\tau+\beta c)/(1/\tau+\beta c)$. It rises from $b$ at $c=0$ to $(b/\tau+\beta)/(1/\tau+\beta)$
  at $c=1$, about 0.53 with the default values. When τ is short compared with growth, the signal tracks
  this curve, so growing density raises the signal, which raises growth further.

**Clinical and biological track.** The second fact is the central message of the notebook. Suppose
serial scans show a tumour growing faster than expected. That could be because it sits in active cortex
(one-way coupling), or simply because it is an intrinsically faster tumour. In this model, both give
*exactly* the same growth curve; size measurements cannot distinguish them, however many there are.
Only a measurement of the activity itself, or an experiment that changes it, can.

**Modeling track.** With β = 0 and $a(0)=b$, $a\equiv b$ is the solution of the signal equation, and the
density equation becomes logistic with rate $\rho(1+\kappa b)$. The model is therefore structurally
non-identifiable from density: only the product $\kappa b$ enters, so every pair $(\kappa, b)$ with the
same product, and every $\rho$ with the same $\rho(1+\kappa b)$, gives the same density curve. The
nullcline follows from $da/dt=0$ by solving a linear equation in $a$. Early in growth, when $c$ is
small, the nullcline is close to $b$, so feedback and one-way runs start almost identically; they
separate only as density rises.

The next cell solves the well-mixed model numerically (fourth-order Runge–Kutta, 0.1-day steps) and
compares it with the first two exact results.

In [ ]:
run = feedback_scenario()
days = run["runs"]["baseline"]["time_days"]
pick = [np.argmin(np.abs(days - d)) for d in (30, 60, 90, 120)]
exact_baseline = logistic_density(0.05, 0.04, days)
np.testing.assert_allclose(run["runs"]["baseline"]["density"], exact_baseline, rtol=1e-9)
np.testing.assert_allclose(run["runs"]["one_way"]["density"], run["effective_logistic"], rtol=1e-9)
rows = [
    f"| {days[i]:.0f} | {exact_baseline[i]:.3f} | {run['runs']['baseline']['density'][i]:.3f} | "
    f"{run['effective_logistic'][i]:.3f} | {run['runs']['one_way']['density'][i]:.3f} | "
    f"{run['runs']['feedback']['density'][i]:.3f} | {run['runs']['feedback']['signal'][i]:.3f} |"
    for i in pick
]
display(
    Markdown(
        "**Table 1. Well-mixed density against exact solutions "
        "(c0 = 0.05, b = 0.3; synthetic).**\n\n"
        "| Day | Baseline, logistic | Baseline, solver | One-way, logistic with ρ_eff | "
        "One-way, solver | Feedback c | Feedback a |\n"
        "| --- | --- | --- | --- | --- | --- | --- |\n" + "\n".join(rows)
    )
)
print(f"ρ_eff = 0.04 × (1 + 1.5 × 0.3) = {run['effective_rate_per_day']:.3f} per day")

**How to read Table 1.** Each row is one day. The second and third columns compare the exact logistic
curve with the solver for the baseline; the fourth and fifth do the same for the one-way case with
$\rho_{\text{eff}}=0.058$ per day. The cell asserts that each pair agrees to a relative tolerance of
1e-9, which is why they look identical. The last two columns give the feedback density and its signal.

**Interpretation.** Starting from 5% density, the baseline reaches 0.15 by day 30 and 0.87 by day 120.
The imposed signal of 0.3 makes one-way growth markedly faster (0.23 at day 30, 0.98 at day 120): half
the carrying capacity is reached on day 51 instead of day 74. Feedback adds comparatively little on top
of that (0.24 at day 30, half the carrying capacity on day 48), but its signal climbs from 0.30 to 0.36
at day 30, 0.465 at day 60 and 0.53 at day 120, while the one-way signal stays at 0.30. In this
parameter setting, *most* of the acceleration comes from the imposed activity; the loop adds a smaller
amount that grows as the tumour fills in.

In [ ]:
figure = plt.figure(figsize=(14, 4.3))
draw_feedback_state(figure, run)
show_figure(figure)
feedback = run["runs"]["feedback"]
gap = np.max(np.abs(feedback["signal"] - signal_nullcline(feedback["density"], 0.3, 0.12, 4.0)))
assert gap < 0.05
print(f"Largest distance of the feedback trajectory from the signal nullcline: {gap:.3f}")

**Figure 1. Three nested assumptions without space (synthetic).** Left: density for baseline, one-way
and feedback. The dotted black curve is the exact logistic with $\rho_{\text{eff}}=\rho(1+\kappa b)$;
it lies exactly on the one-way curve. Middle: the signal index stays at the drive (0.3) unless feedback
is present. Right: the feedback trajectory in the (c, a) plane, starting at the black dot and running
just below the dashed signal nullcline; grey arrows show the direction of change.

**Reading the panels.**
- *Left:* three S-shaped curves. The one-way curve (and the dotted logistic on top of it) is shifted
  about 23 days earlier than the baseline; the feedback curve is shifted a further 3 days. No feature
  of the one-way curve reveals that a signal is involved.
- *Middle:* the flat one-way and baseline lines at 0.3 are the drive. The feedback signal rises as
  density rises and levels off near 0.53, the nullcline value at full density.
- *Right:* the trajectory begins at density 0.05 and signal 0.3 and moves right and up. It stays within
  0.02 of the dashed nullcline (the cell asserts less than 0.05), because the signal relaxes in 4 days
  while density changes over weeks. The arrows show that, off the nullcline, the signal is pulled back
  to it much faster than density changes.

**Interpretation.** In a well-mixed setting, the loop behaves as if the growth rate slowly increases as
the tumour grows. That is a qualitative signature, but on its own a modest one: it would have to be
separated from many other reasons why growth curves deviate from a logistic.

## 4. Explore the well-mixed model

**Why explore.** The figure shows one setting. The explorer lets you change each coefficient and see
which features move and which stay fixed. For each task, first *predict* what will happen, then move the
sliders, then *explain* what you see. Moving a slider needs a running kernel; in a static rendering only
the initial state is shown.

Sliders: κ (signal → growth), β (tumour → signal, 1/day), the external drive $b$, the relaxation time τ
(days) and the initial density $c_0$. Things to try:
1. Set β to 0 and change κ and b. The one-way curve always stays on the dotted ρ_eff logistic: imposed
   activity and faster growth cannot be told apart from density alone.
2. Increase τ to 20 days. The feedback trajectory now lags up to 0.10 below its nullcline: the signal
   cannot keep up with growth. Yet density reaches half the carrying capacity sooner (day 45 instead of
   48), because a slower return to the drive lets the tumour hold the signal higher: the nullcline now
   reaches about 0.79 at full density.
3. Set κ to 0 but keep β. The signal still rises, to about 0.52 by day 150, yet density is unchanged.
   A changing signal is not evidence that it drives growth.
4. Set b to 0. Without feedback the signal is 0 and the coupling has nothing to act on: one-way equals
   baseline. With feedback the tumour creates its own signal (about 0.32 by day 150) and reaches half
   the carrying capacity on day 66 instead of 74.

**Clinical and biological track.** Task 3 is a warning about correlation: a tumour can raise activity,
for example causing seizures, without that activity feeding back into growth. Observing
hyperexcitability around a glioma does not by itself establish that it promotes growth; that needs a
perturbation, as in the cited animal experiments. Task 4 corresponds to a tumour in a quiet region that
generates its own growth-promoting activity.

**Modeling track.** Task 1 is the structural non-identifiability of section 3. Task 2 shows that the
quasi-steady-state picture (the signal on its nullcline) fails when τ is not short compared with the
growth time scale $1/\rho$ = 25 days, and that τ affects not only the lag but also the nullcline
itself.

In [ ]:
from glioma_ai.biology_interactive import feedback_explorer

feedback_explorer()

## 5. Back to space: the controlled comparison

On the constructed plane the tumour starts as a Gaussian centred at x = −5 mm (peak density 0.35,
width 4 mm), and the external drive is a Gaussian bump of amplitude 0.7 centred at (4, 2) mm (width
7 mm) on a background of 0.1. The figure below compares baseline and feedback after 20 days.

**Clinical and biological track.** Picture a tumour beside, but not centred on, a highly active
cortical region. Over 20 days it spreads and grows; with coupling, its edge that faces the active region
is pushed outwards faster. With feedback, the tumour also raises activity in the tissue it occupies.

**Modeling track.** Now the signal varies in space through $b(x,y)$ and, with feedback, through $c$.
Because the signal does not diffuse, the feedback raises $a$ only where density is present. The code
cell checks that the baseline equals the plain reaction–diffusion solver exactly, that the integrals are
ordered feedback > one-way > baseline, and that halving the time step changes the result by less
than 1%.

In [ ]:
display(Image(filename=str(output / "neuron_glioma.png")))
coordinates, initial, drive = neuron_glioma_construction()
parameters = NeuronGliomaParameters()
baseline_parameters = replace(parameters, signal_growth_coupling=0, tumor_signal_feedback_per_day=0)
baseline = simulate_neuron_glioma(initial, drive, drive, baseline_parameters, 1, 1, 0.1, 200)
reference = integrate_fisher_2d(initial, 0.2, 0.04, 1, 1, 0.1, 200)
np.testing.assert_array_equal(baseline["density"], reference)

values = summary["final_density_integral_mm2"]
assert values["feedback"] > values["one_way"] > values["baseline"]
assert summary["step_halving_relative_integral_difference"] < 0.01
summary["step_halving_relative_integral_difference"]

**Figure 2. Synthetic neuron–glioma model after 20 prescribed days.** Top: density for the baseline
and the feedback case. Bottom left: how much feedback raised the signal above the drive; it is highest
where the tumour is, not where the drive is. Bottom right: the density integral (mm² on this 2D plane;
neither cell number nor clinical volume) for the three nested assumptions. The baseline is identical to
the plain reaction–diffusion solver, and halving the time step changes the feedback integral by less
than 1%.

**Reading the panels.**
- *Top:* both tumours spread from the starting point at x = −5 mm. The feedback tumour is larger and
  denser (peak 0.54 against 0.40), and its extra density is shifted towards the drive (centred near
  x = −2.6 mm).
- *Bottom left:* the increase of the signal above the drive peaks at about 0.13, at (−7, 0) mm, inside
  the tumour and slightly away from the drive. There the drive is low, so the factor $(1-a)$ leaves the
  most room for the tumour to raise it.
- *Bottom right:* starting from 35.2 mm², the integral reaches 67.3 mm² (baseline), 93.0 mm² (one-way)
  and 95.8 mm² (feedback) at day 20. Feedback is 1.42 times the baseline but only 1.03 times the
  one-way case. Halving the time step changes the feedback integral by about 0.1%.

**Interpretation.** Over 20 days, most of the extra growth comes from the imposed activity; the loop
adds about 3%. The signal map shows *where* the loop acts: within the tumour, not at the peak of the
drive.

## 6. Sensitivity is not parameter estimation

The next grid varies κ and β with all other inputs fixed. Its ratios describe the chosen constructions.
Setting κ = 0 removes the effect of the signal on density, even if the signal itself changes.

Different assumptions can generate similar density trajectories. The second check repeats the
well-mixed confounding identity in space: a one-way model with constant signal 0.3 is identical to a
baseline with $\rho_{\text{eff}}=0.04\times(1+1.5\times0.3)$ per day.

**Clinical and biological track.** A sensitivity grid answers "how much would it matter if the coupling
were weaker or stronger?", not "how strong is it in patients?". The confounding check is the spatial
version of section 3: from tumour images alone, a tumour in uniformly active tissue and an intrinsically
faster tumour look the same.

**Modeling track.** The grid is a local exploration of a prescribed construction. Parameter estimation
would require data, an observation model linking them to $c$ and $a$, a noise model, and an
identifiability analysis ([Raue et al. 2009](https://doi.org/10.1093/bioinformatics/btp358)); with density data alone, the confounding
identity shows that κ, $b$ and ρ cannot be separated.

In [ ]:
display(Image(filename=str(output / "neuron_glioma_sensitivity.png")))
ratios = np.array(summary["sensitivity"]["integral_ratio_to_baseline"])
np.testing.assert_allclose(ratios[:, 0], 1, rtol=0, atol=1e-12)

# Confounding example: constant one-way signal is absorbed into effective rho.
constant_drive = np.full_like(initial, 0.3)
one_way_parameters = replace(parameters, tumor_signal_feedback_per_day=0)
effective_parameters = replace(
    baseline_parameters,
    proliferation_per_day=0.04 * (1 + 1.5 * 0.3),
)
one_way = simulate_neuron_glioma(
    initial, constant_drive, constant_drive, one_way_parameters, 1, 1, 0.1, 100
)
effective = simulate_neuron_glioma(
    initial, constant_drive, constant_drive, effective_parameters, 1, 1, 0.1, 100
)
np.testing.assert_allclose(one_way["density"], effective["density"], rtol=1e-12, atol=1e-14)
"The same density trajectory does not identify the biological explanation."

**Figure 3. Synthetic sensitivity, not a patient effect.** Final density integral relative to
baseline for three values each of κ and β. The left column (κ = 0) is exactly 1 whatever β is.

**Reading the grid.** Columns are κ = 0, 0.75 and 1.5; rows are β = 0, 0.06 and 0.12 per day. Along
the bottom row (β = 0, one-way), the ratio rises from 1 to 1.18 and 1.38 with κ. Moving up a column
adds feedback: at κ = 1.5, β = 0.12 gives 1.42 instead of 1.38. The ratio depends much more on κ than
on β over 20 days, because feedback needs density to build up before it acts.

## 7. Explore the spatial model

**Why explore.** Section 5 used one drive position. The spatial explorer reruns all three cases on the
plane (61 × 61 grid, 0.1-day steps; a few tenths of a second per change) so that you can move the
activity relative to the tumour. Sliders: κ, β, the drive amplitude, the drive centre x (mm) and the
displayed day. The left panel shows feedback density with the baseline's c = 0.1 contour dashed and the
drive in cyan contours, the middle panel the signal raised by feedback, and the right panel the three
integrals over time. Things to try:
1. Move the drive centre from +4 mm to −5 mm, on top of the tumour, then to +15 mm. On top of the
   tumour the one-way integral rises to 105.9 mm² and feedback adds 2%; far away it falls to
   75.8 mm² and feedback adds 4.5%.
2. Set the drive amplitude to 0. One-way and baseline now differ only through the 0.1 background
   (73.6 against 67.3 mm²), while feedback still raises the signal where the tumour is (77.1 mm²).
3. Set κ to 0. The signal map still changes (by up to 0.10), but all three integrals coincide at
   67.3 mm².
4. Compare day 10 and day 40 with the default settings. The ratio of feedback to one-way grows from
   1.014 to 1.056.

**Clinical and biological track.** Task 1 says that, in this model, a tumour next to active cortex
benefits most from imposed activity, while a tumour in a quiet region relies relatively more on the
activity it generates itself. Task 4 says that the loop matters more the longer it has to act. Neither
is a statement about any patient; both are questions an experiment could address.

**Modeling track.** The relative contribution of feedback is largest where the drive is small, because
the factor $(1-a)$ leaves more room for the tumour to raise the signal and because the one-way
reference is then weaker. Task 3 is the spatial form of "κ = 0 decouples density from the signal".

In [ ]:
from glioma_ai.biology_interactive import spatial_explorer

spatial_explorer()

## 8. Connecting to our recent segmentation work

Our [published imaging pipeline](https://doi.org/10.1080/03091902.2026.2713703) and
[public code](https://github.com/arvidl/high-precision-glioma-segmentation) provide tumour compartments
and subject-specific regional-burden profiles (notebook 09). Those profiles suggest **where to
investigate** an interaction; they neither measure this model's signal index nor estimate its coupling
coefficients.

A proposed bridge would retain reviewed native-space measurements, spatial correspondence and
uncertainty, tissue or physiological evidence, treatment context, and an explicit observation model.
Converting MRI label volume into normalized tumour density would require justification. We perform none
of those conversions or patient-specific fits here.

## 9. What would make this clinically informative?

**Clinical and biological track.**
- Structural MRI can supply reviewed geometry and longitudinal measurements.
- EEG/MEG, intraoperative electrocorticography or invasive recordings can characterize aspects of
  neural activity; none directly measures the abstract index used here.
- Spatial tissue assays, patch-clamp recordings in resected tissue and functional experiments are
  needed to investigate cell states, synaptic interactions and causal mechanisms. Retrograde tracing
  maps which neurons connect to a tumour (Tetzlaff et al. 2025; Sun et al. 2025).
- Controlled experimental platforms are needed to perturb one arrow at a time. Mature rat brain
  organoids with myelinated neurons, synapses and glial cells allow real-time measurement of
  glioblastoma invasion, including single-cell invasion speed, and of therapeutic interventions
  ([Zhou et al. 2026](https://doi.org/10.1093/neuonc/noaf271)). *Proposal, not done by those authors or here:* such a platform could
  in principle be used to measure density and an activity readout together while activity is
  manipulated, which is the kind of data section 3 shows to be necessary.
- Subtype-specific clinical evidence is needed. The levetiracetam association in DMG was retrospective
  and absent in hemispheric high-grade glioma (Barron et al. 2026).
- Calibration, held-out evaluation and patient-relevant endpoints would be required for a predictive
  clinical application.

Functional connectivity is not interchangeable with synapse count. Structural MRI alone does not
establish neuronal activity or macrophage states. Results from a particular high-grade glioma setting
need explicit subtype boundaries. Several cited studies tested drugs that act on the loop in animal or
cell models (ADAM10 inhibitors, perampanel, sulfasalazine, gabapentin, lorazepam, levetiracetam, an
mTOR inhibitor, a RAC1 inhibitor, a DRD1 antagonist), with effects that in some cases differed by
subtype; **no treatment recommendation follows from this simulation.**

**Modeling track.** To move from a construction to a testable model, one would need an observation
model for each data type (for example, how an electrophysiological measure relates to $a$, and how a
label volume relates to $c$), measurements of both the signal and the density over time, a
perturbation that changes one of them, and an identifiability analysis before any fit. Section 3 shows
why density alone is not enough.

## 10. Summary and exercises

**What this notebook shows.**
- *A loop can be written down with explicit, testable limits.* The baseline, one-way and feedback
  models are nested, and the baseline reproduces the plain reaction–diffusion and logistic solutions
  exactly (Table 1, section 5).
- *Imposed activity is confounded with faster growth.* With one-way coupling, the density curve is
  exactly a logistic with $\rho_{\text{eff}}=\rho(1+\kappa b)$ (Figure 1; section 6).
- *Feedback adds growth that builds up with density.* The tumour raises its own signal, which then
  tracks the signal nullcline; in the default settings this adds about 3% to the one-way integral after
  20 days and more later (Figures 1–2, spatial explorer).
- *Where activity is imposed matters.* The feedback acts within the tumour, and its relative
  contribution is largest where the external drive is weak (Figure 2, spatial explorer).

**What it does not show.**
- *That such feedback exists or matters in a patient.* The cited studies motivate the structure; the
  model neither tests nor reproduces them.
- *Any coefficient.* All values are prescribed for teaching.
- *Any relation to measured activity, connectivity, seizures or cognition*, or any MRI appearance.
- *Any treatment effect.*
- *Growth-restraining or subtype-dependent input.* The coupling can only promote growth, although
  inhibitory circuits restrained adult gliomas in recent experiments and GABA's effect differed between
  subtypes (section 1).

**Exercises: clinical and biological track.**
1. *Seizures and growth.* A patient's glioma presents with seizures. Which arrow of the loop do seizures
   reflect? Using the well-mixed explorer (task 3), explain why seizures alone do not show that activity
   drives growth, and which kind of evidence in the cited studies addresses that second arrow.
2. *Location.* Using the spatial explorer, compare a drive centred on the tumour with one 20 mm away.
   How might this relate to tumours in or near highly active cortex, and which measurement would you
   need before drawing any clinical conclusion?
3. *Which measurement?* List the measurements in section 9 and, for each, say whether it informs $c$,
   $a$, κ or β in this model, or none of them directly.
4. *Boundaries.* McAlpine et al. found more excitable neurons in high-grade than in low-grade glioma
   tissue. How would you express a grade difference in this model, and why would you need separate
   evidence for each tumour type rather than one set of coefficients?
5. *Same transmitter, opposite effect.* GABAergic input promoted DMG growth (Barron et al. 2025) but
   restrained adult glioma (Uesaka et al. 2026). What mechanism did Barron et al. identify for the
   difference in DMG? What would this imply for interpreting the levetiracetam association of Barron
   et al. (2026) in other glioma types, and why can a retrospective association not settle it?

**Exercises: modeling track.**
1. *Equivalence class.* In the well-mixed model with β = 0, find all pairs (κ, b) that give the same
   density curve as the default one-way case. How many are there, and what single combination of
   parameters does density identify?
2. *Nullcline.* Derive the signal nullcline and show that at $c=1$ it equals
   $(b/\tau+\beta)/(1/\tau+\beta)$. Check the value 0.53 for the defaults and 0.79 for τ = 20 days.
   Can the signal ever reach 1?
3. *Early growth.* Show that for small $c$ the feedback and one-way models have the same initial growth
   rate $\rho(1+\kappa b)$. Using the explorer or Table 1, estimate when the difference in density
   first exceeds 10%.
4. *Distinguishing data.* Which additional measurement, taken together with density, would distinguish
   one-way coupling from feedback in this model? Would a single time point suffice? Sketch an
   identifiability argument.
5. *A restraining signal.* Add an inhibitory index $g$ with its own drive and relaxation, and replace
   $(1+\kappa a)$ by $(1+\kappa a-\lambda g)$. Which constraint on the parameters keeps the growth factor
   positive? Which nested special cases does the extended model contain, and which new confounding
   identity appears when $a$ and $g$ are both constant?

## Assistance and acceptance evidence

This implementation was developed with Codex and expanded on 10 and 13 October 2026 with a Cursor
agent. The repository's actual assistant-development summary records the model choice, limiting-case
checks and unresolved calibration. It remains pending human scientific review; a checkable record does
not measure AI benefit. See `docs/ai-use/2026-10-06-biology-modeling.md` and notebook 04.

## References

All links were checked on 13 October 2026; the summaries in this notebook follow the published
abstracts.

**Cancer neuroscience: activity drives glioma growth**
- Venkatesh HS et al. Neuronal activity promotes glioma growth through neuroligin-3 secretion. *Cell*
  2015. [doi:10.1016/j.cell.2015.04.012](https://doi.org/10.1016/j.cell.2015.04.012)
- Venkatesh HS et al. Targeting neuronal activity-regulated neuroligin-3 dependency in high-grade
  glioma. *Nature* 2017. [doi:10.1038/nature24014](https://doi.org/10.1038/nature24014)
- Venkatesh HS et al. Electrical and synaptic integration of glioma into neural circuits. *Nature*
  2019. [doi:10.1038/s41586-019-1563-y](https://doi.org/10.1038/s41586-019-1563-y)
- Venkataramani V et al. Glutamatergic synaptic input to glioma cells drives brain tumour progression.
  *Nature* 2019. [doi:10.1038/s41586-019-1564-x](https://doi.org/10.1038/s41586-019-1564-x)
- Osswald M et al. Brain tumour cells interconnect to a functional and resistant network. *Nature*
  2015. [doi:10.1038/nature16071](https://doi.org/10.1038/nature16071)

**Many inputs, inhibition and subtype (2024–2026)**
- Tetzlaff SK et al. Characterizing and targeting glioblastoma neuron-tumor networks with retrograde
  tracing. *Cell* 2025. [doi:10.1016/j.cell.2024.11.002](https://doi.org/10.1016/j.cell.2024.11.002)
- Sun Y et al. Brain-wide neuronal circuit connectome of human glioblastoma. *Nature* 2025.
  [doi:10.1038/s41586-025-08634-7](https://doi.org/10.1038/s41586-025-08634-7)
- Anastasaki C et al. Aberrant coupling of glutamate and tyrosine kinase receptors enables neuronal
  control of brain-tumor growth. *Neuron* 2025. [doi:10.1016/j.neuron.2025.08.005](https://doi.org/10.1016/j.neuron.2025.08.005)
- Xue Z et al. The dopamine receptor D1 inhibitor, SKF83566, suppresses GBM stemness and invasion
  through the DRD1-c-Myc-UHRF1 interactions. *Journal of Experimental & Clinical Cancer Research* 2024.
  [doi:10.1186/s13046-024-02947-7](https://doi.org/10.1186/s13046-024-02947-7)
- Barron T et al. GABAergic neuron-to-glioma synapses in diffuse midline gliomas. *Nature* 2025.
  [doi:10.1038/s41586-024-08579-3](https://doi.org/10.1038/s41586-024-08579-3)
- Barron T et al. Levetiracetam therapeutically targets GABAergic synapses in diffuse midline glioma.
  *Nature Medicine* 2026. [doi:10.1038/s41591-026-04646-6](https://doi.org/10.1038/s41591-026-04646-6)
- Uesaka N et al. Inhibitory circuits restrain adult glioma progression by suppressing calcium-driven
  oncogenic programs. *Neuron* 2026. [doi:10.1016/j.neuron.2026.08.008](https://doi.org/10.1016/j.neuron.2026.08.008)

**Gliomas raise neural activity**
- Buckingham SC et al. Glutamate release by primary brain tumors induces epileptic activity. *Nature
  Medicine* 2011. [doi:10.1038/nm.2453](https://doi.org/10.1038/nm.2453)
- Yu K et al. PIK3CA variants selectively initiate brain hyperactivity during gliomagenesis. *Nature*
  2020. [doi:10.1038/s41586-020-1952-2](https://doi.org/10.1038/s41586-020-1952-2)
- Hatcher A et al. Pathogenesis of peritumoral hyperexcitability in an immunocompetent CRISPR-based
  glioblastoma model. *Journal of Clinical Investigation* 2020.
  [doi:10.1172/jci133316](https://doi.org/10.1172/jci133316)
- Zhang Y et al. Potassium ion channel modulation at cancer-neural interface enhances neuronal
  excitability in epileptogenic glioblastoma multiforme. *Neuron* 2025.
  [doi:10.1016/j.neuron.2024.10.016](https://doi.org/10.1016/j.neuron.2024.10.016)
- Goldberg AR et al. Glioma-induced alterations in excitatory neurons are reversed by mTOR inhibition.
  *Neuron* 2025. [doi:10.1016/j.neuron.2024.12.026](https://doi.org/10.1016/j.neuron.2024.12.026)
- Ding C et al. Glioblastoma-secreted C1QL1 orchestrates tumor microtube expansion and neural synaptic
  pruning to drive malignant synapse formation and recurrence. *Cancer Discovery* 2026.
  [doi:10.1158/2159-8290.CD-25-1123](https://doi.org/10.1158/2159-8290.CD-25-1123)

**Tumour networks, their partners and experimental platforms (Bergen and collaborators)**
- Joseph JV et al. TGF-β promotes microtube formation in glioblastoma through thrombospondin 1.
  *Neuro-Oncology* 2022. [doi:10.1093/neuonc/noab212](https://doi.org/10.1093/neuonc/noab212)
- Watson DC et al. GAP43-dependent mitochondria transfer from astrocytes enhances glioblastoma
  tumorigenicity. *Nature Cancer* 2023. [doi:10.1038/s43018-023-00556-5](https://doi.org/10.1038/s43018-023-00556-5)
- Storevik S et al. Mitochondria transfer in glioblastoma. *Neuro-Oncology* 2026.
  [doi:10.1093/neuonc/noag038](https://doi.org/10.1093/neuonc/noag038)
- Zhou W et al. Development of a highly differentiated rat brain organoid model for exploring
  glioblastoma invasion dynamics and therapy. *Neuro-Oncology* 2026. [doi:10.1093/neuonc/noaf271](https://doi.org/10.1093/neuonc/noaf271)
- Baxter SM et al. Use of psychotropic medications among glioma patients in Denmark, Norway, Sweden,
  and Wales. *Journal of Neuro-Oncology* 2025. [doi:10.1007/s11060-025-04996-0](https://doi.org/10.1007/s11060-025-04996-0)

**Human evidence**
- Krishna S et al. Glioblastoma remodelling of human neural circuits decreases survival. *Nature* 2023.
  [doi:10.1038/s41586-023-06036-1](https://doi.org/10.1038/s41586-023-06036-1)
- McAlpine H et al. Increased neural excitability and glioma synaptic activity drives glioma
  proliferation in human cortex. *Nature Neuroscience* 2026 (online 2025).
  [doi:10.1038/s41593-025-02149-0](https://doi.org/10.1038/s41593-025-02149-0)

**Reviews and classification**
- Monje M et al. Roadmap for the emerging field of cancer neuroscience. *Cell* 2020.
  [doi:10.1016/j.cell.2020.03.034](https://doi.org/10.1016/j.cell.2020.03.034)
- Winkler F et al. Cancer neuroscience: state of the field, emerging directions. *Cell* 2023.
  [doi:10.1016/j.cell.2023.02.002](https://doi.org/10.1016/j.cell.2023.02.002)
- Louis DN et al. The 2021 WHO Classification of Tumors of the Central Nervous System: a summary.
  *Neuro-Oncology* 2021. [doi:10.1093/neuonc/noab106](https://doi.org/10.1093/neuonc/noab106)

**Modeling**
- Swanson KR et al. Virtual and real brain tumors: using mathematical modeling to quantify glioma growth
  and invasion. *Journal of the Neurological Sciences* 2003. [doi:10.1016/j.jns.2003.06.001](https://doi.org/10.1016/j.jns.2003.06.001)
- Raue A et al. Structural and practical identifiability analysis of partially observed dynamical
  models by exploiting the profile likelihood. *Bioinformatics* 2009.
  [doi:10.1093/bioinformatics/btp358](https://doi.org/10.1093/bioinformatics/btp358)